<a href="https://colab.research.google.com/github/udayRage/cuPAMI/blob/main/notebooks/frequentPattern/cuFPGrowth.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding frequent pattern using cuFPGrowth

This tutorial shows how to prepare a dataset, run cuFPGrowth, inspect the generated patterns, and save the results.

[Algorithm documentation](https://github.com/udayRage/cuPAMI/blob/main/docs/algorithms/cuFPGrowth.md)

# Prerequisites

1. Installing cuPAMI and CuPy

Select a GPU runtime in Colab. The following cell installs missing packages. The CuPy package shown is for CUDA 12.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cupami") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/udayRage/cuPAMI.git"])
if importlib.util.find_spec("cupy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "cupy-cuda12x"])

2. Checking the GPU

In [2]:
import cupy as cp

print("CUDA devices:", cp.cuda.runtime.getDeviceCount())


CUDA devices: 2


3. Preparing a sample dataset

Use BinaryTransactionalDatabase with a fixed seed so the input can be reproduced.

In [3]:
from cupami import BinaryTransactionalDatabase, cuFPGrowth

generator = BinaryTransactionalDatabase(transactions=100, items=8, avg_length=3, seed=42)
data = generator.create()
print(data[:5])


[[2, 5, 6, 7], [1, 2, 4, 5], [1], [2, 4, 7], [1, 2, 7]]


## Finding patterns using cuFPGrowth

### Step 1: Choosing the input parameters

`minSup=0.1` means 10% of transactions.

### Step 2: Executing cuFPGrowth

In [4]:
miner = cuFPGrowth(data, minSup=0.1)
miner.mine()


### Step 3: Investigating the generated patterns

Display the number of patterns and the first ten results.

In [5]:
from itertools import islice

patterns = miner.getPatterns()
print("Patterns:", len(patterns))
for pattern, support in islice(patterns.items(), 10):
    print(pattern, support)


Patterns: 35
4 29
4	5 10
3	4 10
1	4 11
4	8 12
2	4 12
4	6 13
3 34
3	5 12
3	7 13


### Step 4: Saving patterns and measuring runtime

Save the patterns to `patterns.txt`. Loading, mining, and total runtime are reported in seconds.

In [6]:
print("Loading:", miner.getLoadingTime())
print("Mining:", miner.getMiningTime())
print("Total:", miner.getTotalRuntime())
miner.save("patterns.txt")


Loading: 8.02499707788229e-05
Mining: 0.31095784180797637
Total: 0.3110380917787552
